# Piper TTS Custom Voice Training
Train your own private Piper TTS voice model on a free Google Colab GPU (T4), then export to  for lightning-fast CPU playback in Paraclea.

In [ ]:
# @title 1. Install Dependencies & Clone Piper Training Tools
!apt-get update -qq && apt-get install -y -qq espeak-ng git-lfs
!pip install --upgrade -q pip
!git clone https://github.com/rhasspy/piper.git
%cd piper/src/python_training
!pip install -q -r requirements.txt
print("Environment ready!")

In [ ]:
# @title 2. Download Pretrained Medium Checkpoint (Transfer Learning)
!mkdir -p checkpoints
%cd checkpoints
!wget -c -q -O en_US-lessac-medium.ckpt "https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.ckpt"
%cd ..
print("Base checkpoint downloaded!")

In [ ]:
# @title 3. Upload & Unpack Your Voice Dataset
# Upload your dataset zip containing wavs/ and metadata.csv
from google.colab import files
import os, zipfile
uploaded = files.upload()
for fn in uploaded.keys():
    if fn.endswith(".zip"):
        with zipfile.ZipFile(fn, "r") as z:
            z.extractall("/content/dataset")
print("Dataset extracted to /content/dataset")

In [ ]:
# @title 4. Preprocess Dataset to LJSpeech Format
!python3 -m piper_train.preprocess \n  --language en \n  --input-dir /content/dataset \n  --output-dir /content/training_dir \n  --dataset-format ljspeech \n  --single-speaker \n  --sample-rate 22050
print("Preprocessing complete!")

In [ ]:
# @title 5. Start Training
!python3 -m piper_train \n  --dataset-dir /content/training_dir \n  --accelerator gpu \n  --devices 1 \n  --batch-size 16 \n  --validation-split 0.05 \n  --max_epochs 1000 \n  --resume_from_checkpoint /content/piper/src/python_training/checkpoints/en_US-lessac-medium.ckpt \n  --checkpoint-epochs 50 \n  --precision 32

In [ ]:
# @title 6. Export to ONNX for Paraclea CPU Playback
import glob
ckpts = sorted(glob.glob("/content/training_dir/lightning_logs/**/checkpoints/*.ckpt", recursive=True))
latest_ckpt = ckpts[-1] if ckpts else "/content/training_dir/last.ckpt"
print(f"Exporting from: {latest_ckpt}")
!python3 -m piper_train.export_onnx {latest_ckpt} /content/diana_piper.onnx
from google.colab import files
files.download("/content/diana_piper.onnx")
files.download("/content/diana_piper.onnx.json")